# MAGIC Telescope PCA (gamma vs hadron) – Solution

**Short name (GitHub):** `PCATel`

Worked answers for `PCATel_Practice_Skeleton.ipynb`. Numbers below are from `data/telescope_data.csv` (19,020 × 10 + class; 12,332 `g` / 6,688 `h`; 0 NA).

**Headline results**
- `fConc`–`fConc1` \(r \approx 0.976\); size/length/width block \(r \approx 0.70\)–\(0.77\).
- Ordered eigenvalue shares (%): 42.24, 15.75, 10.12, 9.94, 7.42, 6.50, 4.08, 2.20, 1.55, 0.20.
- Cumulative: PC2 58.0%, PC4 78.1%, **PC7 96.1%** (95% budget).
- LinearSVC, 33% test, `random_state=42`: **2 PCs 0.742** vs **first 2 raw features 0.719** vs majority 0.648 vs all 10 features 0.787.

Not an IACT trigger and not a discovery claim.


## Inline cheat-sheet (keep this cell visible)

See **`PCATel_Cheatsheet.docx`**. Same table as the skeleton.

**Order:** clean → split label → correlate → eigen / PCA → choose \(k\) → project → classify → simulate.


## Desired outcome

![flowchart](pcatel_flowchart.png)


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from PCATel import HILLAS, load_telescope, standardize, eigen_from_corr, project, reconstruction_mse

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7.2, 4.4)


## 1. Why PCA on MAGIC?

The ten Hillas parameters are linear combinations of the same shower image. A heatmap shows *which* pairs share variance. PCA then spends its first axes on those shared directions instead of treating each column as new information.

Task 1 answer: *because PCA is a rotation of the covariance/correlation structure — if the heatmap is nearly diagonal, PCA will not compress anything useful.*


In [ ]:
# Task 1
print("PCA compresses shared variance. The heatmap is the map of that sharing.")


## 2. Observing the dataset

In [ ]:
df = pd.read_csv("data/telescope_data.csv", index_col=0)
print("NA cells before drop:", int(df.isna().sum().sum()))
df = df.dropna()
print("shape", df.shape)
print(df["class"].value_counts())
print(df.head(3))

classes = df["class"]
data_matrix = df.drop(columns="class")
print("data_matrix", data_matrix.shape, "columns", list(data_matrix.columns))


## 3. Correlation heatmap

In [ ]:
correlation_matrix = data_matrix.corr()
fig, ax = plt.subplots(figsize=(8.0, 6.4))
im = ax.imshow(correlation_matrix, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(HILLAS))); ax.set_yticks(range(len(HILLAS)))
ax.set_xticklabels(HILLAS, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(HILLAS, fontsize=8)
for i in range(len(HILLAS)):
    for j in range(len(HILLAS)):
        val = correlation_matrix.iloc[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6.2,
                color="white" if abs(val) > 0.65 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Task 3 — Pearson correlation of Hillas parameters")
plt.show()
print("fConc vs fConc1 r =", round(correlation_matrix.loc["fConc", "fConc1"], 3))
print("fLength vs fWidth r =", round(correlation_matrix.loc["fLength", "fWidth"], 3))


## 4. Eigendecomposition

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(correlation_matrix.values)
print("Eigenvalues length:", eigenvalues.size, "n_features:", data_matrix.shape[1])
indices = eigenvalues.argsort()[::-1]
eigenvalues = np.real(eigenvalues[indices])
eigenvectors = np.real(eigenvectors[:, indices])
print("sorted eigenvalues:", eigenvalues)
print("shapes", eigenvalues.shape, eigenvectors.shape)


## 5. Scree

In [ ]:
information_proportions = eigenvalues / eigenvalues.sum()
information_percents = information_proportions * 100
print("information %:", np.round(information_percents, 2))

plt.figure()
plt.plot(np.arange(1, 11), information_percents, "ro-", linewidth=2)
plt.title("Task 5: Scree plot")
plt.xlabel("Principal Axes")
plt.ylabel("Percent of Information Explained")
plt.xticks(range(1, 11)); plt.grid(alpha=0.3)
plt.show()


## 6. Cumulative information

In [ ]:
cumulative_information_percents = np.cumsum(information_percents)
print("cumulative %:", np.round(cumulative_information_percents, 2))
k95 = int(np.searchsorted(cumulative_information_percents, 95) + 1)
print("smallest k with cum >= 95%:", k95)

plt.figure()
plt.plot(np.arange(1, 11), cumulative_information_percents, "ro-", linewidth=2)
plt.hlines(y=95, xmin=1, xmax=10, colors="tab:blue", linestyles="--")
plt.vlines(x=k95, ymin=0, ymax=100, colors="tab:green", linestyles=":")
plt.title("Task 6: Cumulative information percentages")
plt.xlabel("Principal Axes")
plt.ylabel("Cumulative Proportion of Variance Explained")
plt.xticks(range(1, 11)); plt.ylim(0, 105); plt.grid(alpha=0.3)
plt.show()


## 7. Standardize + sklearn PCA

In [ ]:
mean = data_matrix.mean(axis=0)
sttd = data_matrix.std(axis=0)
data_matrix_standardized = (data_matrix - mean) / sttd
print(data_matrix_standardized.head(2))

pca = PCA()
principal_components = pca.fit_transform(data_matrix_standardized)
print("n features original / PC space:", data_matrix.shape[1], principal_components.shape[1])

singular_values = pca.singular_values_
eigenvalues_sk = singular_values ** 2
eigenvectors_sk = pca.components_.T
print("first 3 sklearn eigenvalues (from SV^2):", eigenvalues_sk[:3])
print("explained_variance_ (sklearn) first 3:", pca.explained_variance_[:3])

principal_axes_variance_ratios = pca.explained_variance_ratio_
principal_axes_variance_percents = principal_axes_variance_ratios * 100
print("sklearn %:", np.round(principal_axes_variance_percents, 2))
print("matches NumPy scree?", np.allclose(principal_axes_variance_percents, information_percents, atol=0.05))


## 8. Two-component scatter

In [ ]:
pca2 = PCA(n_components=2)
Z2 = pca2.fit_transform(data_matrix_standardized)
print("PC features", Z2.shape[1], "original", data_matrix_standardized.shape[1])
print("2-PC variance ratios", np.round(pca2.explained_variance_ratio_, 3))

fig, ax = plt.subplots(figsize=(7.4, 5.2))
g = classes.values == "g"
ax.scatter(Z2[g, 0], Z2[g, 1], s=8, alpha=0.35, c="#2471a3", label="g  gamma")
ax.scatter(Z2[~g, 0], Z2[~g, 1], s=8, alpha=0.35, c="#c0392b", label="h  hadron")
ax.set_xlabel("PC1 (42.2%)")
ax.set_ylabel("PC2 (15.8%)")
ax.set_title("Task 12 — first two PCs, hue = class")
ax.legend(markerscale=3); ax.grid(alpha=0.25)
plt.show()
print("Overlap is large: PCA is unsupervised and variance != class separation.")


## 9. LinearSVC: 2 PCs vs first 2 original features

In [ ]:
y = classes.astype("category").cat.codes
print("code map", dict(zip(classes.astype("category").cat.categories, range(2))))

pca_1 = PCA(n_components=2)
X = pca_1.fit_transform(data_matrix_standardized)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
svc_1 = LinearSVC(random_state=0, tol=1e-5, max_iter=8000)
svc_1.fit(X_train, y_train)
score_1 = svc_1.score(X_test, y_test)
print(f"Score for model with 2 PCA features: {score_1:.4f}")

first_two_original_features = [0, 1]
X_original = data_matrix_standardized.iloc[:, first_two_original_features]
X_train, X_test, y_train, y_test = train_test_split(X_original, y, test_size=0.33, random_state=42)
svc_2 = LinearSVC(random_state=0, max_iter=8000)
svc_2.fit(X_train, y_train)
score_2 = svc_2.score(X_test, y_test)
print(f"Score for model with 2 original features: {score_2:.4f}")
print(f"Majority-class baseline (always g): {(classes=='g').mean():.4f}")
print("The 2-PC model wins: those two axes are the highest-variance *combinations*, not just fLength and fWidth.")


## 10. Alternate code

In [ ]:
# A. SVD path
X_std = data_matrix_standardized.values
U, S, Vt = np.linalg.svd(X_std, full_matrices=False)
ratios_svd = (S ** 2) / (S ** 2).sum()
print("A SVD %:", np.round(100 * ratios_svd, 2))
Z_svd = X_std @ Vt[:2].T
print("A 2-PC SVD vs sklearn close?", np.allclose(np.abs(Z_svd), np.abs(Z2), atol=1e-6) or
      np.corrcoef(Z_svd[:, 0], Z2[:, 0])[0, 1] > 0.999)

# B. StandardScaler (population std, ddof=0)
Xs_sk = StandardScaler().fit_transform(data_matrix)
print("B scaler vs manual max |diff|:", np.max(np.abs(Xs_sk - X_std)))

# C. covariance of the standardized matrix (should match correlation of raw)
evals_c, evecs_c, pct_c, cum_c = eigen_from_corr(data_matrix)
print("C helper %:", np.round(pct_c, 2))

# D. variance budget API
pca95 = PCA(n_components=0.95).fit(data_matrix_standardized)
print("D PCA(n_components=0.95) kept", pca95.n_components_, "axes")


## 11. More practice

In [ ]:
cum = np.cumsum(pca.explained_variance_ratio_) * 100
for thr in (80, 90, 95, 99):
    k = int(np.searchsorted(cum, thr) + 1)
    print(f"P1  {thr}% needs k={k}  (cum={cum[k-1]:.2f})")

print("P2 PC1 loadings")
for name, w in zip(HILLAS, pca.components_[0]):
    print(f"   {name:10s} {w:+.3f}")
print("P2 PC2 loadings")
for name, w in zip(HILLAS, pca.components_[1]):
    print(f"   {name:10s} {w:+.3f}")
print("PC1 = size/length/width vs concentration. PC2 = fM3Long / fAsym vs fAlpha.")

ks = list(range(1, 11))
mses = [reconstruction_mse(X_std, eigenvectors_sk, k) for k in ks]
plt.figure()
plt.plot(ks, mses, "o-")
plt.xlabel("k"); plt.ylabel("MSE on standardized X")
plt.title("P3 reconstruction error vs k")
plt.xticks(ks); plt.grid(alpha=0.3)
plt.show()

print("P4 LinearSVC vs k")
accs = []
for k in ks:
    Pk = PCA(n_components=k).fit_transform(data_matrix_standardized)
    Xtr, Xte, ytr, yte = train_test_split(Pk, y, test_size=0.33, random_state=42)
    acc = LinearSVC(random_state=0, tol=1e-5, max_iter=8000).fit(Xtr, ytr).score(Xte, yte)
    accs.append(acc)
    print(f"   k={k:2d}  acc={acc:.4f}")

X_drop = data_matrix_standardized.drop(columns="fConc1")
pca_drop = PCA().fit(X_drop)
print("P5 without fConc1, first %:", np.round(100 * pca_drop.explained_variance_ratio_[:4], 2))


## 12. Simulation

In [ ]:
def run_once(k=2, n=None, noise_sigma=0.0, scale=True, seed=0, test_size=0.33):
    rng = np.random.default_rng(seed)
    Xb = data_matrix.values.copy()
    yb = y.values.copy()
    if n is not None and n < len(Xb):
        idx = rng.choice(len(Xb), size=n, replace=False)
        Xb, yb = Xb[idx], yb[idx]
    if scale:
        mu = Xb.mean(axis=0)
        sd = Xb.std(axis=0, ddof=1)
        sd[sd == 0] = 1.0
        Xb = (Xb - mu) / sd
    if noise_sigma > 0:
        Xb = Xb + rng.normal(0.0, noise_sigma, size=Xb.shape)
    k_use = min(k, Xb.shape[1], Xb.shape[0])
    Z = PCA(n_components=k_use).fit_transform(Xb)
    Xtr, Xte, ytr, yte = train_test_split(Z, yb, test_size=test_size, random_state=seed)
    return float(LinearSVC(random_state=0, max_iter=8000).fit(Xtr, ytr).score(Xte, yte))

print("sweep k")
for k in range(1, 11):
    print(" ", k, round(run_once(k=k, seed=42), 4))

print("sweep n at k=2")
for n in (500, 1000, 4000, 10000, 19020):
    print(" ", n, round(run_once(k=2, n=n, seed=1), 4))

print("sweep noise at k=2")
for sig in (0.0, 0.25, 0.5, 1.0, 1.5):
    print(" ", sig, round(run_once(k=2, noise_sigma=sig, seed=7), 4))

print("scale off vs on, k=2")
print("  scale True ", round(run_once(k=2, scale=True, seed=3), 4))
print("  scale False", round(run_once(k=2, scale=False, seed=3), 4))


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence on this card |
|----------|----------------|---------------------------|
| Expert (IACT) | loadings + \(\lambda\) spectrum | PC1 is size vs concentration; 7 PCs keep 96.1% of Hillas variance. |
| Technician | scale, `n_components`, seed | Standardize with training \(\mu,\sigma\), `PCA(n_components=7)`, persist `V`. |
| Executive | one comparison | Two PCs beat length+width on a linear screen: 0.742 vs 0.719 (majority 0.648). |
| Nonspecialist | picture | Ten correlated ellipse numbers become a few summary axes; signal and background still overlap. |


## What this model can and cannot do

**Can:** compress correlated Hillas parameters; draw a 2-D unsupervised map; feed a linear classifier with orthogonal axes.

**Cannot:** replace MAGIC's physics G/H discriminator; peek at `class` while building axes; treat high variance as class-relevant; support a flux or discovery claim.

Full 10-feature LinearSVC is ~0.787. PCA is a pre-processor, not the analysis.


## Next steps

See the skeleton. Rebuild the executive sentence from `PCATel_Project_Memo.docx` without pasting the loading table.
